# DERBi-PIE-Pokorny (DPP) Analysis

Two questions drive this notebook:

1. **Orthography** — Does DPP use updated laryngeal notation (h₁, h₂, h₃)? If yes, we can replace IELex etyma outright with DPP's forms instead of just gap-filling.
2. **Coverage** — What gaps in IELex (etyma with no reflexes, missing etyma entirely) can DPP fill?

Data paths (relative to repo root):
- DPP: `data/ielex/DERBi-PIE-Pokorny/pok_all(Sheet1)_utf8.csv`
- IELex: `data/ielex/current_ielex/lex_*.csv`

In [ ]:
import re
import json
from pathlib import Path
import pandas as pd

REPO = Path("../../../")  # src/ielex/notebooks/ → repo root
DPP_FILE = REPO / "data/ielex/DERBi-PIE-Pokorny/pok_all(Sheet1)_utf8.csv"
IELEX_DIR = REPO / "data/ielex/current_ielex"

dpp = pd.read_csv(DPP_FILE, encoding="utf-8-sig")
etyma = pd.read_csv(IELEX_DIR / "lex_etyma.csv")
reflex = pd.read_csv(IELEX_DIR / "lex_reflex.csv")
etyma_reflex = pd.read_csv(IELEX_DIR / "lex_etyma_reflex.csv")
language = pd.read_csv(IELEX_DIR / "lex_language.csv")

print(f"DPP rows:          {len(dpp):>7,}")
print(f"IELex etyma:       {len(etyma):>7,}")
print(f"IELex reflexes:    {len(reflex):>7,}")
print(f"Etymon-reflex links: {len(etyma_reflex):>5,}")
print(f"Languages:         {len(language):>7,}")

DPP rows:           67,656
IELex etyma:         2,222
IELex reflexes:     63,116
Etymon-reflex links: 64,301
Languages:             379


/var/folders/qp/sgyn9rjs7jl_zv5cksz_txj40000gn/T/ipykernel_25096/3231603514.py:10: DtypeWarning: Columns (1,9) have mixed types. Specify dtype option on import or set low_memory=False.
  dpp = pd.read_csv(DPP_FILE, encoding="utf-8-sig")


## 1. DPP Dataset Overview

In [2]:
print("DPP columns:", list(dpp.columns))
dpp.head(10)

DPP columns: ['pok_all_index', 'pok_wd_index', 'web_root', 'root', 'abbr', 'reflex', 'meaning', 'notes (specific)', 'notes (general/overflow)', 'notes (shared/overflow)', 'root_norm', 'web_root_norm', 'combined_roots']


,pok_all_index,pok_wd_index,web_root,root,abbr,reflex,meaning,notes (specific),notes (general/overflow),notes (shared/overflow),root_norm,web_root_norm,combined_roots
0,0,0,ā,ā,ai.,ā,Ausruf der Besinnung,NaN,NaN,NaN,ā,ā,ā | ā
1,0,1,ā,ā,gr.,ἆ,"Ausruf des Unwillens, Schmerzes, Erstaunens",NaN,NaN,NaN,ā,ā,ā | ā
2,0,2,ā,ā,gr.,ἆ,Ausruf der Verwunderung und Klage,NaN,NaN,NaN,ā,ā,ā | ā
3,0,3,ā,ā,gr.,ἀά,Ausruf der Verwunderung und Klage,NaN,NaN,NaN,ā,ā,ā | ā
4,0,4,ā,ā,gr.,ἄζειν,ächzen,NaN,NaN,NaN,ā,ā,ā | ā
5,0,5,ā,ā,lat.,ā,"Ausruf des Schmerzes, des Unwillens",NaN,NaN,NaN,ā,ā,ā | ā
6,0,6,ā,ā,lat.,āh,"Ausruf des Schmerzes, des Unwillens",NaN,NaN,NaN,ā,ā,ā | ā
7,0,7,ā,ā,lit.,à,"Ausruf der Verwunderung, des Tadels oder Spottes",NaN,NaN,NaN,ā,ā,ā | ā
8,0,8,ā,ā,lit.,aà,"Ausruf der Verwunderung, des Tadels oder Spottes",NaN,NaN,NaN,ā,ā,ā | ā
9,0,9,ā,ā,lit.,ā,Ausruf der verwunderten Frage,lauter Neuschöpfungen,NaN,NaN,ā,ā,ā | ā


In [3]:
# Basic shape: unique roots vs total reflex rows
n_unique_roots = dpp["root"].nunique()
n_unique_web_roots = dpp["web_root"].nunique()
n_unique_abbr = dpp["abbr"].nunique()

print(f"Unique 'root' values:     {n_unique_roots:,}")
print(f"Unique 'web_root' values: {n_unique_web_roots:,}")
print(f"Unique language abbrs:    {n_unique_abbr:,}")
print(f"\nRows with empty 'root':   {dpp['root'].isna().sum() + (dpp['root'] == '').sum()}")
print(f"Rows with empty 'reflex': {dpp['reflex'].isna().sum() + (dpp['reflex'] == '').sum()}")

Unique 'root' values:     1,517
Unique 'web_root' values: 2,209
Unique language abbrs:    1,789

Rows with empty 'root':   7072
Rows with empty 'reflex': 662


In [4]:
# Top 20 languages by reflex count
(
    dpp.groupby("abbr")
    .size()
    .sort_values(ascending=False)
    .head(20)
    .rename("reflex_count")
    .reset_index()
)

,abbr,reflex_count
0,gr.,6022
1,ai.,4271
2,lat.,2926
3,lit.,2199
4,ahd.,2172
5,ags.,2049
6,aisl.,1714
7,lit.,1648
8,ags.,1437
9,ahd.,1368


## 2. Orthography Check — Does DPP Use Updated Laryngeal Notation?

Modern IE notation inserts laryngeals as **h₁, h₂, h₃** (Unicode subscripts) or sometimes **H₁, H₂, H₃** or plain **H**.  
Pokorny's original notation has none of these — he wrote reconstructed PIE without laryngeal theory applied.

We check the `root`, `web_root`, and `root_norm` fields.

In [5]:
# Unicode subscript laryngeals
LARYNGEAL_PATTERNS = {
    "h₁ (U+2081)": "h\u2081",
    "h₂ (U+2082)": "h\u2082",
    "h₃ (U+2083)": "h\u2083",
    "H₁": "H\u2081",
    "H₂": "H\u2082",
    "H₃": "H\u2083",
    "plain 'H'": r"\bH\b",   # word-boundary capital H (regex)
    "h1 / h2 / h3 (ASCII)": r"h[123]",
}

for field in ["root", "web_root", "root_norm"]:
    text = dpp[field].fillna("")
    print(f"\n--- {field} ---")
    for label, pat in LARYNGEAL_PATTERNS.items():
        count = text.str.contains(pat, regex=True, na=False).sum()
        if count:
            print(f"  {label}: {count:,} rows")


--- root ---

--- web_root ---

--- root_norm ---


In [6]:
# Show examples of any capital-H hits so we can judge if they're laryngeals
h_in_root = dpp[dpp["root"].fillna("").str.contains(r"H", regex=False)]
print(f"Rows where 'root' contains 'H': {len(h_in_root)}")
if not h_in_root.empty:
    print("\nSample:")
    print(h_in_root[["root", "web_root"]].drop_duplicates(subset="root").head(15).to_string(index=False))

Rows where 'root' contains 'H': 123

Sample:
                                      root                                 web_root
2. pet-, petə- : ptē-, ptō-, Hellenic ptā- pet-2, petə- : ptē-, ptō- (griech. ptā-)


In [7]:
# Same check on IELex etyma for baseline comparison
print("IELex etyma — laryngeal check:")
ielex_entries = etyma["entry"].fillna("")
for label, pat in LARYNGEAL_PATTERNS.items():
    count = ielex_entries.str.contains(pat, regex=True, na=False).sum()
    if count:
        print(f"  {label}: {count:,} rows")
else:
    # Python 'for...else' fires when loop completes without break
    print("  (none found — IELex uses classic Pokorny notation throughout)")

IELex etyma — laryngeal check:
  (none found — IELex uses classic Pokorny notation throughout)


### Orthography finding

Summarize what the cells above tell us. Expected result: **neither DPP nor IELex currently uses laryngeal notation** — both are straight Pokorny. The H hits in `root` are likely language-name abbreviations (e.g. Greek *ptā-* → "Hellenic ptā-"), not laryngeal symbols. This means:

- DPP does **not** give us a free orthography upgrade.
- Declan's hand-applied corrections remain the only source of updated laryngeal forms so far.
- The case for outright replacement (rather than gap-filling) based on orthography **does not hold** for now.

## 3. Root Alignment — DPP Roots vs IELex Etyma

DPP `root` and IELex `entry` both encode Pokorny etyma, but they may use slightly different forms.
We try three match strategies:

1. **Exact** — `root` == `entry`
2. **Normalized** — strip whitespace, diacritics on punctuation, lowercase
3. **Prefix / substring** — root is a prefix of entry (IELex often lists variant forms separated by commas)

In [8]:
def normalize(s: str) -> str:
    """Lowercase, collapse whitespace, strip leading/trailing punctuation noise."""
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s

dpp_roots = dpp[["root", "root_norm"]].drop_duplicates(subset="root").copy()
dpp_roots["root_norm_clean"] = dpp_roots["root"].apply(normalize)

etyma_clean = etyma[["id", "entry"]].copy()
etyma_clean["entry_norm"] = etyma_clean["entry"].apply(normalize)

# --- Strategy 1: exact normalized match ---
exact_matches = dpp_roots.merge(
    etyma_clean, left_on="root_norm_clean", right_on="entry_norm", how="inner"
)
print(f"Exact normalized matches:    {len(exact_matches):,} DPP roots match an IELex etymon")

# --- Strategy 2: IELex entry starts with DPP root (prefix match) ---
# Build lookup: normalized root → list of IELex etyma that start with it
dpp_root_set = set(dpp_roots["root_norm_clean"].unique())

prefix_hits = 0
prefix_map = {}  # ielex entry_norm → matched dpp root
for _, row in etyma_clean.iterrows():
    entry = row["entry_norm"]
    for root in dpp_root_set:
        if root and entry.startswith(root):
            prefix_map[entry] = root
            prefix_hits += 1
            break

print(f"Prefix matches (IELex starts with DPP root): {prefix_hits:,} IELex etyma")

Exact normalized matches:    817 DPP roots match an IELex etymon
Prefix matches (IELex starts with DPP root): 830 IELex etyma


In [9]:
# Show sample exact matches
print("Sample exact matches (DPP root → IELex entry):")
print(exact_matches[["root", "entry"]].head(15).to_string(index=False))

Sample exact matches (DPP root → IELex entry):
                                         root                                         entry
                                            ā                                             ā
                                          ab-                                           ab-
                                         abh-                                          abh-
                                       abhro-                                        abhro-
                                       abō(n)                                        abō(n)
                                 ades-, ados-                                  ades-, ados-
                                         ag̑-                                          ag̑-
ā̆g̑her-, ā̆g̑hen-, ā̆g̑hes-, or ō̆g̑her etc. ā̆g̑her-, ā̆g̑hen-, ā̆g̑hes-, or ō̆g̑her etc.
                                         agh-                                          agh-
                                 

In [10]:
# How many IELex etyma have at least one DPP root aligned (by either strategy)?
exact_etyma_ids = set(exact_matches["id"].tolist())
prefix_etyma_ids = set(
    etyma_clean.loc[etyma_clean["entry_norm"].isin(prefix_map.keys()), "id"].tolist()
)
aligned_ids = exact_etyma_ids | prefix_etyma_ids
print(f"IELex etyma with a DPP root aligned (exact or prefix): {len(aligned_ids):,} / {len(etyma):,}")
print(f"IELex etyma with no DPP alignment:                     {len(etyma) - len(aligned_ids):,}")

IELex etyma with a DPP root aligned (exact or prefix): 830 / 2,222
IELex etyma with no DPP alignment:                     1,392


## 4. Gap Analysis — IELex Etyma with No Reflexes

In [11]:
etyma_with_reflexes = set(etyma_reflex["etyma_id"].unique())
empty_etyma = etyma[~etyma["id"].isin(etyma_with_reflexes)].copy()

print(f"Etyma with ≥1 reflex:   {len(etyma_with_reflexes):,}")
print(f"Etyma with 0 reflexes:  {len(empty_etyma):,}")
print(f"  ({100*len(empty_etyma)/len(etyma):.1f}% of all etyma are empty)")
empty_etyma[["id", "entry", "gloss"]].head(15)

Etyma with ≥1 reflex:   1,541
Etyma with 0 reflexes:  681
  (30.6% of all etyma are empty)


,id,entry,gloss
3,3952,"(s)k<sup>u̯</sup>el-, (s)kel-","{""en"":""to call, resound""}"
4,3966,(s)k<sup>u̯</sup>er-,"{""en"":""to do, make""}"
7,3942,"(s)kand-, (s)kend-","{""en"":""to glow; bright""}"
12,3953,(s)k̑el-,"{""en"":""to jump, spring""}"
17,3959,(s)kep-,"{""en"":""to hide, cover""}"
19,3961,(s)ker-,"{""en"":""to shrink, dry out, wrinkle up; dry, th..."
21,3965,(s)ker-,"{""en"":""(onomatopoeic: hoarse sound)""}"
24,3969,"(s)ker-dh-, (s)k<sub>o</sub>r-dh-","{""en"":""small, pitiful, miserable""}"
26,3970,(s)kert-s-,"{""en"":""across, straight along""}"
27,3973,(s)keu-,"{""en"":""to perform, carry out""}"


In [12]:
# Of the empty etyma, how many align with a DPP root?
empty_ids = set(empty_etyma["id"].tolist())
empty_with_dpp = empty_ids & aligned_ids
empty_without_dpp = empty_ids - aligned_ids

print(f"Empty etyma that align with DPP:    {len(empty_with_dpp):,}")
print(f"Empty etyma with no DPP alignment:  {len(empty_without_dpp):,}")
print()
print("These are the etyma DPP could potentially fill:")
fillable = etyma_clean[etyma_clean["id"].isin(empty_with_dpp)]
fillable[["id", "entry"]].head(20)

Empty etyma that align with DPP:    8
Empty etyma with no DPP alignment:  673

These are the etyma DPP could potentially fill:


,id,entry
96,2259,ā(i)g̑h- : īg̑h-
107,2234,ā̆g̑-
112,2238,agh-(lo-)
161,2289,"ālu-, ālo-"
379,2519,bhren-to-s
761,2866,"ghel-ond-, ghol-n̥d-"
1811,3946,"skē̆i-bh-, nasalized ski-m-bh-"
2141,4365,u̯en-


In [13]:
# For a fillable etymon, how many DPP reflexes are available?
# Join via exact root alignment
fillable_roots = exact_matches[exact_matches["id"].isin(empty_with_dpp)][["id", "entry", "root"]]

dpp_per_fillable = (
    fillable_roots.merge(dpp, on="root", how="left")
    .groupby(["id", "entry"])
    .size()
    .sort_values(ascending=False)
    .rename("dpp_reflex_count")
    .reset_index()
)
print("Top empty etyma by DPP reflex count:")
print(dpp_per_fillable.head(20).to_string(index=False))

Top empty etyma by DPP reflex count:
  id entry  dpp_reflex_count
4365 u̯en-                16


## 5. New Etyma — DPP Roots Not in IELex at All

In [14]:
# DPP roots that don't match any IELex etymon (by exact normalized match)
matched_dpp_roots = set(exact_matches["root_norm_clean"].tolist())
all_dpp_roots = dpp_roots.copy()
unmatched_dpp = all_dpp_roots[~all_dpp_roots["root_norm_clean"].isin(matched_dpp_roots)]

print(f"Unique DPP roots total:              {len(all_dpp_roots):,}")
print(f"DPP roots that match an IELex etymon: {len(matched_dpp_roots):,}")
print(f"DPP roots NOT in IELex (potential new etyma): {len(unmatched_dpp):,}")
print()
print("Sample unmatched DPP roots:")
print(unmatched_dpp[["root"]].head(20).to_string(index=False))

Unique DPP roots total:              1,518
DPP roots that match an IELex etymon: 816
DPP roots NOT in IELex (potential new etyma): 702

Sample unmatched DPP roots:
                                root
              ā̆bel-, ā̆bō̆l-, abel-
                              1. ad-
                              2. ad-
                                 NaN
                     agu̯(e)sī, aksī
                          agu̯h-no-s
                              3. ai-
                             4. ā̆i-
                        5. ai- : oi-
             1. aig-, nasalized ing-
                             2. aig-
                             3. aig-
                             aigu̯h-
                             1. ais-
                             2. ais-
                       2. ak̑-, ok̑-
aku̯ā-, more accurately əku̯ā, ēku̯-
                         1. al-, ol-
                              2. al-
                             3. ā̆l-


In [15]:
# How many reflexes do unmatched roots have? (indicates potential addition size)
unmatched_root_vals = set(unmatched_dpp["root"].tolist())
unmatched_reflexes = dpp[dpp["root"].isin(unmatched_root_vals)]
print(f"Total reflex rows for unmatched roots: {len(unmatched_reflexes):,}")
print()
# Distribution of reflex counts per unmatched root
reflex_dist = (
    unmatched_reflexes.groupby("root")
    .size()
    .describe()
)
print(reflex_dist)

Total reflex rows for unmatched roots: 41,891

count    701.000000
mean      49.670471
std       64.142931
min        1.000000
25%       16.000000
50%       29.000000
75%       56.000000
max      641.000000
dtype: float64


## 6. Language Abbreviation Mapping

DPP uses German-style abbreviations (`ai.` = Altindisch, `gr.` = Greek, `lat.` = Latin).  
IELex uses English-style abbreviations (`OInd`, `Gr`, `Lat`).  
Any import pipeline will need a mapping between them.

In [16]:
# Show all DPP abbreviations with their reflex count
dpp_abbrs = (
    dpp.groupby("abbr")
    .size()
    .sort_values(ascending=False)
    .rename("reflex_count")
    .reset_index()
)
print(f"Total unique DPP abbr values (incl. trailing-space variants): {len(dpp_abbrs)}")
print()
print("Top 40 DPP abbreviations:")
print(dpp_abbrs.head(40).to_string(index=False))

Total unique DPP abbr values (incl. trailing-space variants): 1789

Top 40 DPP abbreviations:
  abbr  reflex_count
   gr.          6022
   ai.          4271
  lat.          2926
  lit.          2199
  ahd.          2172
  ags.          2049
 aisl.          1714
 lit.           1648
 ags.           1437
 ahd.           1368
  gr.           1366
   av.          1279
 lett.          1230
aisl.           1166
  air.          1049
  got.          1013
lett.           1009
 lat.            957
  mhd.           838
 aksl.           834
  ai.            822
  arm.           809
 mhd.            748
 cymr.           703
  nhd.           669
aksl.            620
  av.            612
 nhd.            600
anord.           587
 russ.           567
 got.            526
 air.            523
russ.            517
  alb.           506
   as.           480
  mir.           478
cymr.            469
  apr.           459
 mnd.            433
  mnd.           428


In [17]:
# Note: DPP abbr has many trailing-space duplicates — quantify
dpp["abbr_stripped"] = dpp["abbr"].str.strip()
stripped_abbrs = dpp.groupby("abbr_stripped").size().sort_values(ascending=False).reset_index()
print(f"After stripping whitespace: {len(stripped_abbrs)} unique abbreviations")
print(stripped_abbrs.head(30).to_string(index=False))

After stripping whitespace: 1296 unique abbreviations
abbr_stripped    0
          gr. 7394
          ai. 5100
         lat. 3899
         lit. 3851
         ahd. 3551
         ags. 3490
        aisl. 2889
        lett. 2244
          av. 1893
         mhd. 1589
         air. 1575
         got. 1540
        aksl. 1456
         nhd. 1274
        cymr. 1173
        russ. 1085
         arm. 1052
       anord.  944
         mnd.  866
          as.  835
         mir.  822
         apr.  803
         alb.  741
        umbr.  540
        engl.  529
        norw.  498
        gall.  479
        bret.  461
        čech.  373
         osk.  356


In [18]:
# IELex language abbreviations for comparison
print("IELex language abbreviations (sample):")
print(language[["id", "name", "abbr"]].head(30).to_string(index=False))

IELex language abbreviations (sample):
 id                         name  abbr
375           {"en":"Old Irish"}   OIr
376        {"en":"Middle Irish"}   MIr
377               {"en":"Irish"}    Ir
378              {"en":"Gaelic"}  Gael
379        {"en":"Scots Gaelic"} ScotG
380                {"en":"Manx"}  Manx
381              {"en":"Shelta"}  Shel
382          {"en":"Old Breton"} OBret
383       {"en":"Middle Breton"} MBret
384              {"en":"Breton"}  Bret
385         {"en":"Old Cornish"} OCorn
386      {"en":"Middle Cornish"} MCorn
387             {"en":"Cornish"}  Corn
388             {"en":"Cumbric"}  Cumb
389           {"en":"Old Welsh"}    OW
390        {"en":"Middle Welsh"}    MW
391               {"en":"Welsh"}     W
392         {"en":"Celtiberian"}  Celt
393             {"en":"Gaulish"}  Gaul
394 {"en":"Transalpine Gaulish"} TGaul
395   {"en":"Cisalpine Gaulish"} CGaul
396            {"en":"Lepontic"}   Lep
397               {"en":"Noric"} Noric
398            {"en":"Gal

In [19]:
# Naive exact-match between stripped DPP abbrs and IELex abbrs
dpp_abbr_set = set(stripped_abbrs["abbr_stripped"].str.lower())
ielex_abbr_set = set(language["abbr"].str.lower())

shared = dpp_abbr_set & ielex_abbr_set
print(f"Abbreviations shared (case-insensitive):  {len(shared)}")
print(f"DPP abbrs with no IELex match:            {len(dpp_abbr_set - ielex_abbr_set)}")
print(f"IELex abbrs with no DPP match:            {len(ielex_abbr_set - dpp_abbr_set)}")
print()
if shared:
    print("Shared abbreviations:", sorted(shared))

Abbreviations shared (case-insensitive):  15
DPP abbrs with no IELex match:            1279
IELex abbrs with no DPP match:            364

Shared abbreviations: ['alb', 'arm', 'as', 'bret', 'corn', 'hitt', 'ir', 'lat', 'manx', 'mir', 'pali', 'phryg', 'russ', 'serb', 'vlat']


## 7. Summary

Fill in after running cells above.

In [20]:
# Auto-generated summary table
summary = {
    "DPP total reflex rows": len(dpp),
    "DPP unique roots": dpp["root"].nunique(),
    "DPP unique language abbrs (stripped)": dpp["abbr_stripped"].nunique(),
    "IELex etyma (total)": len(etyma),
    "IELex etyma with ≥1 reflex": len(etyma_with_reflexes),
    "IELex etyma with 0 reflexes (gaps)": len(empty_etyma),
    "Empty etyma alignable to DPP": len(empty_with_dpp),
    "DPP roots not in IELex (potential new etyma)": len(unmatched_dpp),
    "DPP reflex rows for potential new etyma": len(unmatched_reflexes),
    "DPP uses laryngeal notation (h₁/h₂/h₃)": "NO — classic Pokorny throughout",
    "IELex uses laryngeal notation": "NO — classic Pokorny throughout",
}

for k, v in summary.items():
    print(f"{k:<50} {v}")

DPP total reflex rows                              67656
DPP unique roots                                   1517
DPP unique language abbrs (stripped)               1296
IELex etyma (total)                                2222
IELex etyma with ≥1 reflex                         1541
IELex etyma with 0 reflexes (gaps)                 681
Empty etyma alignable to DPP                       8
DPP roots not in IELex (potential new etyma)       702
DPP reflex rows for potential new etyma            41891
DPP uses laryngeal notation (h₁/h₂/h₃)             NO — classic Pokorny throughout
IELex uses laryngeal notation                      NO — classic Pokorny throughout
